# Kaggle: two independent T4 workers

In [ ]:
from pathlib import Path
import subprocess
import sys

# Clone your published repository once; replace URL if you use a different owner.
repo = Path('/kaggle/working/drl-context-poi')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/sepehrmustafavi/drl-context-poi.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)
subprocess.run([sys.executable, str(repo / 'scripts/check_environment.py')], cwd=repo, check=True)


Prepare real canonical data and freeze validation-selected settings before running. No dataset or performance result is bundled. Use the full data; the automated fixture tests are software checks only.

In [ ]:
# Prepare data/canonical/{nyc,tky,gowalla_ca}.csv and configs/frozen/*.yaml first.
# For Kaggle input data, copy/link user-provided canonical files into the expected paths.
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'plan', '--suite', 'full',
                '--settings-dir', 'configs/frozen', '--out', 'experiments/final'], cwd=repo, check=True)

# Two independent jobs, one per physical T4; no pooled VRAM or DDP.
workers = []
handles = []
for gpu in ('0', '1'):
    log = open(repo / f'worker_gpu{gpu}.log', 'a')
    handles.append(log)
    workers.append(subprocess.Popen([sys.executable, '-m', 'drlpoi.cli', 'worker',
                    '--plan', 'experiments/final/plan.json', '--gpu', gpu], cwd=repo, stdout=log, stderr=subprocess.STDOUT))
print('Worker PIDs:', [p.pid for p in workers])
print('Monitor worker_gpu*.log and experiments/final/state/*.log.')
# Save the complete runs/ and experiments/ trees using Kaggle output persistence before shutdown.
# Reuse the same files and unchanged plan to resume; confirm dead processes before removing stale locks.
